# Seconday ONS Data Join - IMD Deprivation Domains

This notebook details join steps with IMD Deprivation Domains, i.e. the broad deprivation factors that are combined to evaluate overall deprivation in an area (_as already joined as IMD 2025 deciles_).  

The usefulness of this additional layer of detail has not yet been determined: it may be the case that the domain-level relationships align closely with the already examined FSA Rating-IMD Decile (combined factor) relationship, and no new enrichment is gained from this secondary join.  

With this in mind, I will apply the same Kruskal-Wallis test to each domain factor, and will proceed (or not) on the basis of the findings that result.  

First, I will load and inspect the file, saved to `../data/raw/imd_2025_domains.csv`:

__[English indices of deprivation 2025](https://www.gov.uk/government/statistics/english-indices-of-deprivation-2025)__ - as with the combined deciles data already joined: "Statistics on relative deprivation in small areas in England" - but at domain-level (e.g. Employment, Education & Skills). **File size is approx. 4MB**

In [1]:
# load and check domain-level IMD data:
import pandas as pd

imd_domains = pd.read_csv("../data/raw/imd_2025_domains.csv")

print(imd_domains.shape)
print(imd_domains.columns.tolist())
imd_domains.head()

(33755, 20)
['LSOA code (2021)', 'LSOA name (2021)', 'Local Authority District code (2024)', 'Local Authority District name (2024)', 'Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)', 'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)', 'Income Rank (where 1 is most deprived)', 'Income Decile (where 1 is most deprived 10% of LSOAs)', 'Employment Rank (where 1 is most deprived)', 'Employment Decile (where 1 is most deprived 10% of LSOAs)', 'Education, Skills and Training Rank (where 1 is most deprived)', 'Education, Skills and Training Decile (where 1 is most deprived 10% of LSOAs)', 'Health Deprivation and Disability Rank (where 1 is most deprived)', 'Health Deprivation and Disability Decile (where 1 is most deprived 10% of LSOAs)', 'Crime Rank (where 1 is most deprived)', 'Crime Decile (where 1 is most deprived 10% of LSOAs)', 'Barriers to Housing and Services Rank (where 1 is most deprived)', 'Barriers to Housing and Services Decil

,LSOA code (2021),LSOA name (2021),Local Authority District code (2024),Local Authority District name (2024),Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived),Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs),Income Rank (where 1 is most deprived),Income Decile (where 1 is most deprived 10% of LSOAs),Employment Rank (where 1 is most deprived),Employment Decile (where 1 is most deprived 10% of LSOAs),"Education, Skills and Training Rank (where 1 is most deprived)","Education, Skills and Training Decile (where 1 is most deprived 10% of LSOAs)",Health Deprivation and Disability Rank (where 1 is most deprived),Health Deprivation and Disability Decile (where 1 is most deprived 10% of LSOAs),Crime Rank (where 1 is most deprived),Crime Decile (where 1 is most deprived 10% of LSOAs),Barriers to Housing and Services Rank (where 1 is most deprived),Barriers to Housing and Services Decile (where 1 is most deprived 10% of LSOAs),Living Environment Rank (where 1 is most deprived),Living Environment Decile (where 1 is most deprived 10% of LSOAs)
0,E01000001,City of London 001A,E09000001,City of London,26525,8,33730,10,33708,10,33755,10,33108,10,33698,10,29220,9,244,1
1,E01000002,City of London 001B,E09000001,City of London,31203,10,33669,10,33734,10,33672,10,32574,10,33712,10,32640,10,3702,2
2,E01000003,City of London 001C,E09000001,City of London,25913,8,25167,8,26985,8,30273,9,20719,7,27325,9,30400,10,4540,2
3,E01000005,City of London 001E,E09000001,City of London,14807,5,14836,5,17911,6,15886,5,10458,4,25630,8,11294,4,5403,2
4,E01000006,Barking and Dagenham 016A,E09000002,Barking and Dagenham,10917,4,7519,3,15286,5,11134,4,21901,7,17875,6,2745,1,9479,3


-> 33755 rows matches the expected UK-wide LSOA count, column count matches expected (4 identifiers, 2x overall IMD, 7x pairs for each domain).  

As with the combined deciles, I'll rename columns for easier handling / better readbility before proceeding with the main data join:

In [ ]:
# rename IMD domain dataset columns:

imd_domains.columns = [
    "lsoa21cd", "lsoa21nm",
    "lad_code", "lad_name", # lad = Local Authority District
    "imd_rank", "imd_decile",
    "income_rank", "income_decile",
    "employment_rank", "employment_decile",
    "education_rank", "education_decile",
    "health_rank", "health_decile",
    "crime_rank", "crime_decile",
    "barriers_rank", "barriers_decile",
    "living_env_rank", "living_env_decile",
]

print(imd_domains.columns.tolist())

['lsoa21cd', 'lsoa21nm', 'lad_code', 'lad_name', 'imd_rank', 'imd_decile', 'income_rank', 'income_decile', 'employment_rank', 'employment_decile', 'education_rank', 'education_decile', 'health_rank', 'health_decile', 'crime_rank', 'crime_decile', 'barriers_rank', 'barriers_decile', 'living_env_rank', 'living_env_decile']


In [ ]:
# load main dataset

df = pd.read_csv(
    "../data/processed/fsa_london_establishments_with_imd.csv",
    dtype={"PostCode": str, "postcode_tier": str, "lsoa21cd": str, "RightToReply": str}
        # RightToReply throws DtypeWarning otherwise (mostly empty string w/ occasional free-text values)
)

print(df.shape) # should be (81218, 33)

(81217, 33)


/var/folders/wt/l78_yzjx2mggl4v8h1f028p40000gn/T/ipykernel_47433/2662323415.py:3: DtypeWarning: Columns (0: RightToReply) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [ ]:
# join domain deciles to main dataset:

# domain deciles columns to retain
domain_deciles = [
    "income_decile", "employment_decile", "education_decile",
    "health_decile", "crime_decile", "barriers_decile", "living_env_decile"
]

df = df.merge(
    imd_domains[["lsoa21cd"] + domain_deciles],
    on="lsoa21cd",
    how="left",
)

# inspect post join, show matched rows count
print(df.shape) # should be (81217, 40)
print(df[domain_deciles].notna().sum()) # should show 70462 for each

(81217, 40)
income_decile        70462
employment_decile    70462
education_decile     70462
health_decile        70462
crime_decile         70462
barriers_decile      70462
living_env_decile    70462
dtype: int64


-> resulting DataFrame of 81,217 rows and 40 columns is as expected, with 70,462 domain decile-matched rows also expected count. 